# Modelo

Para ver como cambia la performance del modelo con el tratamiento de los datos primero hay que tener un modelo de partida que optimizar.

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from transformers import AudioFilterResampler, SpectralSubtractor, MelSpectrogramTransformer, SpectrogramPadder, FlattenTransformer, FeatureExtractor

from config import TEST_SIZE, SEED

## Preprocesamiento

1. Resampleo y filtro pasa bajos
2. Sustracción espectral (opcional)

## Espectrogramas

Se suelen usar Mel espectrogramas

1. Mel espectrogramas
2. Padding (agregar tipos)
3. Aplanar

In [ ]:
train_df = pd.read_csv('./dataset/ciclos/filtrado/train_melspectrogram.csv')
test_df = pd.read_csv('./dataset/ciclos/filtrado/test_melspectrogram.csv')

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']
X_test = test_df.drop(columns=['label'])
y_test = test_df['label']

### Random Forest

#### Entrenamiento

In [ ]:

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(7, 10),
    'min_samples_split': randint(10, 25),
    'min_samples_leaf': randint(10, 25)
}

Fine-tuning

El score elegido es auc-roc, podría ser recall o f1.

In [ ]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0024B31647230>, 'min_samples_leaf': <scipy.stats....0024B43B4A0D0>, 'min_samples_split': <scipy.stats....0024B43B4A210>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
0,9,13,22,0.755247
6,9,15,14,0.751883
1,9,20,17,0.750573
4,9,20,17,0.750573
3,8,12,16,0.748423
7,8,17,21,0.745564
9,7,10,21,0.745168
8,8,15,11,0.744546
5,7,13,17,0.743842
2,7,14,16,0.739070


In [ ]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 9, 'min_samples_leaf': 13, 'min_samples_split': 22}
Best CV score: 0.7552474271889341


#### Evaluación

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))
print('AUC-ROC: ', round(roc_auc_score(y_test, y_pred), 2))
print('Recall: ', round(recall_score(y_test, y_pred), 2))

              precision    recall  f1-score   support

           0       0.72      0.53      0.61       429
           1       0.67      0.82      0.73       493

    accuracy                           0.68       922
   macro avg       0.69      0.67      0.67       922
weighted avg       0.69      0.68      0.68       922

AUC-ROC:  0.67
Recall:  0.82


Agrego el modelo al pipeline y entreno con los audios de entrenamiento

In [ ]:
df_paths = pd.read_csv('./dataset/ciclos/filtrado/train.csv')
paths = ['./dataset/ciclos/' + f for f in df_paths['cycle_wav_file']]
y_train = df_paths['label'].values

In [ ]:
# pipeline con la que se creo el dataset
pipeline_proc = Pipeline([
    ('filter_resample', AudioFilterResampler()),
    ('melspec', MelSpectrogramTransformer()),
    ('pad', SpectrogramPadder()),
    ('flatten', FlattenTransformer()),
    ('scaler', StandardScaler())
])

pipeline_final = Pipeline([
    ('preproc', pipeline_proc),
    ('rf', best_model)
])

pipeline_final.fit(paths, y_train)

,steps,"[('preproc', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('filter_resample', ...), ('melspec', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,sr_target,16000
,highcut,4000
,lowcut,0


Guardo el modelo

In [54]:
joblib.dump(pipeline_final, '../modelos/melspec_filt_rf.pkl')

['../modelos/melspec_filt_rf.pkl']

#### Evaluación en audios propios

Tengo grabaciones que fueron divididas en ciclos con el segmentador. El audio "respiración4_ciclo5" tiene una sibilancia clara, los demás ciclos de ese audio podrían ser positivos también pero no logro escuchar una sibilancia tan clara.

In [55]:
model = joblib.load('./melspec_filt_rf.pkl')

In [ ]:
path = './audios_propios/ciclos_energy_based'
files = os.listdir(path)

df_audios_propios = pd.DataFrame(files, columns=['cycle_wav_file'])

df_audios_propios['label'] = np.where(df_audios_propios['cycle_wav_file'].str.contains('respiracion4_ciclo_5'), 1, 0)

df_audios_propios # audios de validacion

,cycle_wav_file,label
0,respiracion1_ciclo_1.wav,0
1,respiracion1_ciclo_2.wav,0
2,respiracion1_ciclo_3.wav,0
3,respiracion1_ciclo_4.wav,0
4,respiracion1_ciclo_5.wav,0
5,respiracion1_ciclo_6.wav,0
6,respiracion1_ciclo_7.wav,0
7,respiracion2_ciclo_1.wav,0
8,respiracion2_ciclo_2.wav,0
9,respiracion2_ciclo_3.wav,0


In [57]:
archivos = df_audios_propios['cycle_wav_file'].tolist()

X_propios_paths = [os.path.join(path, f) for f in archivos]
y_propios = df_audios_propios['label'].values

Predecimos

In [ ]:
y_propios_pred = model.predict(X_propios_paths)
y_propios_proba_pred = model.predict_proba(X_propios_paths)[:, 1]

In [60]:
print(classification_report(y_propios, y_propios_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98        48
           1       0.33      1.00      0.50         1

    accuracy                           0.96        49
   macro avg       0.67      0.98      0.74        49
weighted avg       0.99      0.96      0.97        49



In [ ]:
df_audios_propios['predicted'] = y_propios_pred
df_audios_propios['probability'] = y_propios_proba_pred.round(2)
df_audios_propios

,cycle_wav_file,label,predicted,probability
0,respiracion1_ciclo_1.wav,0,0,0.36
1,respiracion1_ciclo_2.wav,0,0,0.14
2,respiracion1_ciclo_3.wav,0,0,0.39
3,respiracion1_ciclo_4.wav,0,0,0.11
4,respiracion1_ciclo_5.wav,0,0,0.24
5,respiracion1_ciclo_6.wav,0,0,0.33
6,respiracion1_ciclo_7.wav,0,0,0.30
7,respiracion2_ciclo_1.wav,0,0,0.13
8,respiracion2_ciclo_2.wav,0,0,0.33
9,respiracion2_ciclo_3.wav,0,0,0.30


Parece prometedor, varios ciclos del audio 4 fueron clasificados como anormales.

## Feature Extraction

1. Resample y Filtro pasa bajos
2. Feature Extractor
    + MFCC
    + ZCR
    + Short-Time Energy
    + SC
    + Spectral Roll-off
    + BER
    + Spectral Flatness
3. Scaler

### Random Forest